# Alpha Optimization & Regime Switching Journal

This notebook documents the mathematical concepts, forward steps, and code enhancements as we work to optimize the `equities-mean-reversion-ml` trading system for positive alpha relative to the S&P 500 benchmark.

## Phase 1: Baseline Establishment and Expansion

### The Challenge
The initial baseline system generated an 18% annualized return but underperformed a pure buy-and-hold strategy of the S&P 500 (SPY), generating an alpha of `-0.0688`. The core of the problem is that the market was in a massive bull run over the past two years. A mean-reversion strategy inherently waits for dips, meaning it misses out on continuous upward rallies.

### Our Initial Optimization:
We tackled this by expanding the trading universe from 8 tech stocks to the **Top 20 most mean-reverting stocks** identified by our quantitative screener. 

The screener ranks stocks using metrics like the **Hurst Exponent**, which mathematically identifies whether a time series trends ($H > 0.5$) or mean-reverts ($H < 0.5$):

$$ \text{E} \left[ \frac{R(n)}{S(n)} \right] = C n^H $$

By expanding the universe, the system executed **65 trades** over the 2-year period, significantly increasing capital utilization to 91.6% and generating a solid **0.99 Sharpe Ratio**.

## Phase 2: Alpha Generation via Regime Switching

To generate true alpha, we cannot rigidly stick to mean-reversion in a bull market. We must adapt dynamically. We use a **Gaussian Mixture Model (GMM)** to cluster the market into 3 hidden volatility regimes:

1. **Regime 0 (Low Volatility / Choppy):** Ideal for **Pairs Trading** (Cointegration).
2. **Regime 1 (Normal / Uptrending):** Ideal for **Momentum**.
3. **Regime 2 (High Volatility / Crash):** Ideal for **Cash**.

The probability density function of our 3-component GMM is defined as:
$$ p(x) = \sum_{k=1}^{3} \pi_k \mathcal{N}(x \mid \mu_k, \Sigma_k) $$

In [ ]:
# Code Snippet: Real-Time Regime Detection
from strategy.regime_detector import RegimeDetector
from features.indicators import IndicatorEngine
from data.fetcher import DataFetcher

# 1. Fetch benchmark data
fetcher = DataFetcher()
df_spy = fetcher.fetch_historical("SPY", period="1y")

# 2. Calculate features (like moving averages and volatility)
ind_engine = IndicatorEngine()
df_spy = ind_engine.compute_all(df_spy)

# 3. Fit the GMM and detect the hidden state
detector = RegimeDetector(n_components=3)
detector.fit(df_spy)
regime, confidence = detector.detect_regime(df_spy)

print(f"Current Market Regime: {regime} (Confidence: {confidence:.1%})")